In [ ]:
!pip install datasets transformers -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("gbharti/finance-alpaca")

print(dataset)

README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json:   0%|          | 0.00/42.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 68912
    })
})


In [ ]:
print(dataset["train"][0])

{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 months. The short

In [ ]:
for i in range(3):
    print(dataset["train"][i])
    print("-" * 80)

{'instruction': 'For a car, what scams can be plotted with 0% financing vs rebate?', 'input': '', 'output': "The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 months. The short

In [ ]:
sample = dataset["train"][0]

print("INSTRUCTION:")
print(sample["instruction"])

print("\nOUTPUT:")
print(sample["output"][:300])

INSTRUCTION:
For a car, what scams can be plotted with 0% financing vs rebate?

OUTPUT:
The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car


In [ ]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""
    }

formatted_dataset = dataset["train"].map(format_prompt)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/68912 [00:00<?, ? examples/s]

### Instruction:
For a car, what scams can be plotted with 0% financing vs rebate?

### Response:
The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 months. The shorter length m

In [ ]:
def is_good_example(example):
    return len(example["output"]) < 500

filtered_dataset = dataset["train"].filter(is_good_example)

print("Original dataset size:", len(dataset["train"]))
print("Filtered dataset size:", len(filtered_dataset))

Filter:   0%|          | 0/68912 [00:00<?, ? examples/s]

Original dataset size: 68912
Filtered dataset size: 47685


In [ ]:
formatted_dataset = filtered_dataset.map(format_prompt)

print(formatted_dataset[0]["text"])

Map:   0%|          | 0/47685 [00:00<?, ? examples/s]

### Instruction:
Why does it matter if a Central Bank has a negative rather than 0% interest rate?

### Response:
That is kind of the point, one of the hopes is that it incentivizes banks to stop storing money and start injecting it into the economy themselves. Compared to the European Central Bank investing directly into the economy the way the US central bank has been doing. (The Federal Reserve buying mortgage backed securities) On a country level, individual European countries have tried this before in recent times with no noticeable effect.


In [ ]:
finance_keywords = [
    "stock", "market", "finance", "investment",
    "bank", "revenue", "profit", "earnings",
    "ratio", "debt", "equity", "shares",
    "trading", "economy", "company"
]

def is_finance_related(example):
    text = (
        example["instruction"].lower() +
        " " +
        example["output"].lower()
    )

    return any(keyword in text for keyword in finance_keywords)

finance_dataset = filtered_dataset.filter(is_finance_related)

print("Filtered finance dataset size:", len(finance_dataset))

Filter:   0%|          | 0/47685 [00:00<?, ? examples/s]

Filtered finance dataset size: 6794
